# Uczenie Maszynowe 1 - Klasyfikatory liniowe - perceptron prosty
**Wojciech Olejnik**

In [7]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
import matplotlib.pyplot as plt
import time
import pandas as pd
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

**Klasa bazowa _LinBase_ implementujacą metody wspólne klasyfikatorów liniowych.**

In [8]:
class LinBase(BaseEstimator, ClassifierMixin):
    def __init__(self, coef=None, intercept=None, class_labels=None):
        self.coef_ = coef
        self.intercept_ = intercept
        self.class_labels_ = class_labels

    def margin(self, X, y):
        m = y * (X.dot(self.coef_) + self.intercept_)
        m = m / LA.norm(self.coef_)
        return np.min(m), m

    def decision_function(self, X):
        a, b = self.margin(X, np.ones(X.shape[0]))
        return b

    def predict_proba(self, X):
        a, b = self.margin(X, np.ones((X.shape[0],)))
        return np.array([1 - 1 / (1 + np.exp(-b)), 1 / (1 + np.exp(-b))]).T

    def predict(self, X):
        results = np.sign(X.dot(self.coef_) + self.intercept_)
        results_mapped = self.class_labels_[1 * (results > 0)]
        return results_mapped

    def get_params(self, deep=True):
        pass

    def set_params(self, **parameters):
        for parameter, value in parameters.items():
            setattr(self, parameter, value)
        return self

    def __str__(self):
        return "{}[w={}, b={}]".format(self.__class__.__name__, self.coef_, self.intercept_)

**Klasa Preceptorn , dziedzicząca po klasie LinBase i implementująca algorytm uczenia preceptronu zgodnie z regułą perceptronu**

In [9]:
class Perceptron(LinBase):
    def __init__(self, break_time=1, **kwargs):
        super(Perceptron, self).__init__(**kwargs)
        self.break_time_ = break_time

    def fit(self, X, y):
        m, n = X.shape
        self.class_labels_ = np.unique(y)
        if self.coef_ is not None:
            w = self.coef_
        else:
            w = np.zeros(n)

        if self.intercept_ is not None:
            b = self.intercept_
        else:
            b = 0

        i = 0
        k = 0
        t1 = time.time()
        ci = 0
        print("PERCEPTRON")
        while(k < m):
            ci += 1
            if y[i] * (w.dot(X[i, :]) + b) <= 0:
                w = w + X[i, :] * y[i]
                b = b + y[i]
                k = 0
            else:
                k += 1
            i = (i + 1) % m
            t2 = time.time()
            if t2 - t1 > self.break_time_:
                print("Przekroczony czas!")
                break
        print(f'Time: {time.time() - t1}\n'
              f'Iterations: {ci}\n'
              f'Number of weight changes: {k}')
        print('-------------------------------')
        self.coef_ = w
        self.intercept_ = b

**Klasa AveragePerceptron (uśrednianie decyzji)**

In [10]:
class AveragePerceptron(LinBase):
    def __init__(self, break_time=10, k_max=1000, **kwargs):
        super(AveragePerceptron, self).__init__(**kwargs)
        self.break_time_ = break_time
        self.v_ = None
        self.v_bias_ = None
        self.k_max = k_max

    def fit(self, X, y):
        m, n = X.shape
        self.class_labels_ = np.unique(y)
        if self.coef_ is not None:
            w = self.coef_
        else:
            w = np.zeros(n)

        if self.intercept_ is not None:
            b = self.intercept_
        else:
            b = 0

        self.v_ = 0 * w
        self.v_bias_ = 0 * b
        i = 0
        k = 0
        c = 0
        t1 = time.time()
        ci = 0
        print("AVERAGE PERCEPTRON")
        while(k < self.k_max):
            ci += 1
            if y[i] * (w.dot(X[i, :]) + b) <= 0:
                self.v_ += c * w
                self.v_bias_ += c * b
                w = w + X[i, :] * y[i]
                b = b + y[i]
                k += 1
                c = 1
            else:
                c += 1
            i = (i + 1) % m
            t2 = time.time()
            if t2 - t1 > self.break_time_:
                print("Przekroczony czas!")
                break
        self.v_ += c * w
        self.v_bias_ += c * b
        print(f'Time: {time.time() - t1}\n'
              f'Iterations: {ci}\n'
              f'Number of weight changes: {c}')
        print('-------------------------------')
        self.coef_ = self.v_
        self.intercept_ = self.v_bias_

**VotedPerceptron (uśrednianie wag)**

In [11]:
class VotedPerceptron(LinBase):
    def __init__(self, break_time=10, k_max=1000, **kwargs):
        super(VotedPerceptron, self).__init__(**kwargs)
        self.break_time_ = break_time
        self.k_max = k_max
        self.c = np.zeros(self.k_max + 1)
        self.v_ = None
        self.v_bias_ = None

    def fit(self, X, y):
        m, n = X.shape
        self.class_labels_ = np.unique(y)
        self.v_ = np.zeros((self.k_max + 1, n))
        self.v_bias_ = np.zeros(self.k_max + 1)
        if self.coef_ is not None:
            w = self.coef_
        else:
            w = np.zeros(n)

        if self.intercept_ is not None:
            b = self.intercept_
        else:
            b = 0

        i = 0
        k = 0
        c = 0
        t1 = time.time()
        ci = 0
        print("VOTED PERCEPTRON")
        while(k < self.k_max):
            ci += 1
            if y[i] * (w.dot(X[i, :]) + b) <= 0:
                self.v_[k, :] = w
                self.v_bias_[k] = b
                self.c[k] = c
                w = w + X[i, :] * y[i]
                b = b + y[i]
                k += 1
                c = 1
            else:
                c += 1
            i = (i + 1) % m
            t2 = time.time()
            if t2 - t1 > self.break_time_:
                print("Przekroczony czas!")
                break

        self.v_[k, :] = w
        self.v_bias_[k] = b
        self.c[k] = c
        print(f'Time: {time.time() - t1}\n'
              f'Iterations: {ci}\n'
              f'Number of weight changes: {c}')
        print('-------------------------------')
        self.coef_ = self.v_
        self.intercept_ = self.v_bias_

    def predict(self, X):
        v = np.hstack((self.v_, np.array([self.v_bias_])))
        X = np.vstack((X, np.ones((X.shape[0], 1))))
        return np.sign(np.dot(np.sign(np.dot(v, X)), self.c)).T

    def predict_proba(self, X):
        v = np.hstack((self.v_, np.array([self.v_bias_]).T))
        X = np.hstack((X, np.ones((X.shape[0], 1))))
        a = np.sign(np.dot(np.sign(np.dot(v, X.T)).T, self.c))
        b = 1-a
        return np.vstack((a, b)).T

    def margin(self, X, y):
        return 0, 0

In [12]:
def plotClass(X, y, clf=None, marker=["ro", "k+"], n=50):
    plt.figure()
    uy = np.unique(y)
    for i, u in enumerate(uy):
        plt.plot(X[y == u, 0], X[y == u, 1], marker[i], label=str(u))

    if clf is not None:
        if clf.name_ == 'LinSvm':
            plt.scatter(X[clf.svm_inds, 0], X[clf.svm_inds, 1], s=100,
                        facecolors='none', zorder=10, edgecolors='k')

        # if (isinstance(clf, LinSvm)):
        # plt.plot(self.svm_inds, '*')
        # pass
        # napisac plota dla punktow clf.svm+inds *
        X1, X2 = np.meshgrid(np.linspace(np.min(X[:, 0]), np.max(X[:, 0]), n),
                             np.linspace(np.min(X[:, 1]), np.max(X[:, 1]), n))
        XX = np.array([X1.ravel(), X2.ravel()]).T
        yy = clf.predict_proba(XX)[:, 1]
        yy = np.reshape(yy, X1.shape)
        plt.contour(X1, X2, yy, [0.5])

    else:
        pass